**Install Feast**

In [1]:
!pip -q install feast==0.64.0 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10

In [2]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Upload the Titanic Dataset**

In [3]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [5]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.head())
print(df.shape)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Examine the Data**

In [6]:
print(df.columns)
print(df.info())
print(df.isnull().sum())

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex       

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Create Features- perform simple feature engineering.**

In [7]:
import pandas as pd

# Entity
df["passenger_id"] = df["PassengerId"].astype("int64")

# Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Encode Sex
df["sex_encoded"] = df["Sex"].map({
    "male": 0,
    "female": 1
}).astype("int64")

# Derived feature 1
df["family_size"] = (
    df["SibSp"] +
    df["Parch"] +
    1
).astype("int64")

# Derived feature 2
df["is_alone"] = (
    df["family_size"] == 1
).astype("int64")

# Rename/cast other features
df["pclass"] = df["Pclass"].astype("int64")
df["age"] = df["Age"].astype("float32")
df["fare"] = df["Fare"].astype("float32")

# Target
df["survived"] = df["Survived"].astype("int64")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
df[
    [
        "passenger_id",
        "pclass",
        "sex_encoded",
        "age",
        "fare",
        "family_size",
        "is_alone",
        "survived"
    ]
].head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,passenger_id,pclass,sex_encoded,age,fare,family_size,is_alone,survived
0,1,3,0,22.0,7.250000,2,0,0
1,2,1,1,38.0,71.283302,2,0,1
2,3,3,1,26.0,7.925000,1,1,1
3,4,1,1,35.0,53.099998,2,0,1
4,5,3,0,35.0,8.050000,1,1,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Add Event Timestamps**   Feast models feature values with timestamps so that historical retrieval can perform point-in-time joins.         Titanic is not naturally a time-series dataset, so for this classroom demonstration we create artificial timestamps.

In [9]:
base_time = pd.Timestamp(
    "1912-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        df["passenger_id"],
        unit="s"
    )
)

df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Separate Features and Labels**

In [10]:
feature_df = df[
    [
        "passenger_id",
        "event_timestamp",
        "created_timestamp",
        "pclass",
        "sex_encoded",
        "age",
        "fare",
        "family_size",
        "is_alone"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
label_df = df[
    [
        "passenger_id",
        "event_timestamp",
        "survived"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
print("Feature data:")
display(feature_df.head())

print("Labels:")
display(label_df.head())

Feature data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,passenger_id,event_timestamp,created_timestamp,pclass,sex_encoded,age,fare,family_size,is_alone
0,1,1912-01-01 00:00:01+00:00,1912-01-01 00:00:02+00:00,3,0,22.0,7.250000,2,0
1,2,1912-01-01 00:00:02+00:00,1912-01-01 00:00:03+00:00,1,1,38.0,71.283302,2,0
2,3,1912-01-01 00:00:03+00:00,1912-01-01 00:00:04+00:00,3,1,26.0,7.925000,1,1
3,4,1912-01-01 00:00:04+00:00,1912-01-01 00:00:05+00:00,1,1,35.0,53.099998,2,0
4,5,1912-01-01 00:00:05+00:00,1912-01-01 00:00:06+00:00,3,0,35.0,8.050000,1,1


Labels:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,passenger_id,event_timestamp,survived
0,1,1912-01-01 00:00:01+00:00,0
1,2,1912-01-01 00:00:02+00:00,1
2,3,1912-01-01 00:00:03+00:00,1
3,4,1912-01-01 00:00:04+00:00,1
4,5,1912-01-01 00:00:05+00:00,0


**Create the Feast Repository**

In [13]:
import os

repo_path = "/content/titanic_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

In [14]:
import os

repo_path = "/content/titanic_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

In [15]:
feature_df.to_parquet(
    f"{repo_path}/data/titanic_features.parquet",
    index=False
)

Create feature_store.yaml

In [16]:
feature_store_yaml = """
project: titanic_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

**Define the Entity and Feature View** Create:features.py

In [17]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)


# -----------------------------
# ENTITY
# -----------------------------

passenger = Entity(
    name="passenger",
    join_keys=["passenger_id"],
    description="Titanic passenger"
)


# -----------------------------
# DATA SOURCE
# -----------------------------

titanic_source = FileSource(
    name="titanic_source",
    path="data/titanic_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)


# -----------------------------
# FEATURE VIEW
# -----------------------------

titanic_feature_view = FeatureView(
    name="titanic_features",
    entities=[passenger],

    ttl=timedelta(days=50000),

    schema=[
        Field(name="pclass", dtype=Int64),
        Field(name="sex_encoded", dtype=Int64),
        Field(name="age", dtype=Float32),
        Field(name="fare", dtype=Float32),
        Field(name="family_size", dtype=Int64),
        Field(name="is_alone", dtype=Int64),
    ],

    source=titanic_source,

    online=True
)


# -----------------------------
# FEATURE SERVICE
# -----------------------------

titanic_feature_service = FeatureService(
    name="titanic_survival_service",
    features=[
        titanic_feature_view
    ]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

In [18]:
!find /content/titanic_feast -maxdepth 2 -type f

/content/titanic_feast/data/titanic_features.parquet
/content/titanic_feast/features.py
/content/titanic_feast/feature_store.yaml


**Apply the Feature Definitions**

In [19]:
%cd /content/titanic_feast

/content/titanic_feast


In [20]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [21]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [22]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

**Create the Feast Store Object**

In [25]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/titanic_feast"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [26]:
feature_service = store.get_feature_service(
    "titanic_survival_service"
)

**Retrieve Historical Features**

In [27]:
entity_df = label_df.copy()

display(entity_df.head())

,passenger_id,event_timestamp,survived
0,1,1912-01-01 00:00:01+00:00,0
1,2,1912-01-01 00:00:02+00:00,1
2,3,1912-01-01 00:00:03+00:00,1
3,4,1912-01-01 00:00:04+00:00,1
4,5,1912-01-01 00:00:05+00:00,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
display(training_data.head())

,passenger_id,event_timestamp,survived,pclass,sex_encoded,age,fare,family_size,is_alone
0,1,1912-01-01 00:00:01+00:00,0,3,0,22.0,7.250000,2,0
1,2,1912-01-01 00:00:02+00:00,1,1,1,38.0,71.283302,2,0
2,3,1912-01-01 00:00:03+00:00,1,3,1,26.0,7.925000,1,1
3,4,1912-01-01 00:00:04+00:00,1,1,1,35.0,53.099998,2,0
4,5,1912-01-01 00:00:05+00:00,0,3,0,35.0,8.050000,1,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Train a Model Using Feast Features**

In [30]:
feature_columns = [
    "pclass",
    "sex_encoded",
    "age",
    "fare",
    "family_size",
    "is_alone"
]

In [31]:
X = training_data[feature_columns]

y = training_data["survived"]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [34]:
predictions = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [35]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 77.65 %


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The important point is not the accuracy. The model received its feature matrix through Feast historical feature retrieval.

**Materialize Features to the Online Store**

In [36]:
%cd /content/titanic_feast

!feast materialize \
    1912-01-01T00:00:00 \
    1912-01-02T00:00:00

/content/titanic_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/c

**Retrieve Online Features**

In [37]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "passenger_id": 25
        }
    ]
).to_dict()

In [38]:
print(online_features)

{'passenger_id': [25], 'family_size': [5], 'fare': [21.075000762939453], 'pclass': [3], 'sex_encoded': [1], 'is_alone': [0], 'age': [8.0]}


In [39]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)

,passenger_id,family_size,fare,pclass,sex_encoded,is_alone,age
0,25,5,21.075001,3,1,0,8.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Make an Online Prediction**

In [40]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [41]:
X_online = online_df[
    feature_columns
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [42]:
prediction = final_model.predict(
    X_online
)

print(
    "Predicted survival:",
    prediction[0]
)

Predicted survival: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Retrieve Features for Multiple Passengers**

In [43]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"passenger_id": 10},
        {"passenger_id": 20},
        {"passenger_id": 30},
        {"passenger_id": 40}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)

,passenger_id,family_size,fare,pclass,sex_encoded,is_alone,age
0,10,2,30.070801,2,1,0,14.0
1,20,1,7.225000,3,1,1,28.0
2,30,1,7.895800,3,0,1,28.0
3,40,2,11.241700,3,1,0,14.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [44]:
online_df["predicted_survival"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

display(
    online_df[
        [
            "passenger_id",
            "predicted_survival"
        ]
    ]
)

,passenger_id,predicted_survival
0,10,1
1,20,1
2,30,0
3,40,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
